In [ ]:
import pandas as pd
from num2words import num2words
import os
import spacy
import matplotlib.pyplot as plt
nlp = spacy.load("nl_core_news_md")

In [ ]:
with open("1000woorden.txt", "r", encoding='utf-8') as f:
    words = f.read()
    word_list = [w.strip().lower() for w in words.split(",")] 
    word_set = set(word_list)

In [ ]:

folder = "../6_Apply BEL/abel_bel"
# folder = "../6_Apply BEL/ibis_bel"
# folder = "../6_Apply BEL/ibis_colon_bel"
amount_predictions = []
amount_unique_predictions = []
non_1000_list = []
flist = []
for csv_file in os.listdir(folder):
    file = os.path.join(folder, csv_file)
    df = pd.read_csv(file)
    non_1000 = 0
    amount_predictions.append(df.shape[0])
    amount_unique_predictions.append(len(df["mention"].unique()))
    for mention in df["mention"].unique():
        mention_norm = str(mention).strip().lower()
        if mention_norm not in word_set:
            non_1000 += 1
    df = pd.read_csv(file, index_col=False)
    flist.append(df)
    non_1000_list.append(non_1000)

df_out = pd.concat(flist, axis=0).reset_index()
df_out = df_out.drop("id", axis=1)

In [ ]:
df_out

BEL words  (corresponding to thesis section "Amount of jargon used")

In [ ]:
df1 = pd.read_csv('ABEL_words_syll.csv')
# df1 = pd.read_csv('IBIS_words_syll.csv')
# df1 = pd.read_csv('IBIS_colon_words_syll.csv')
df1["BEL-mentions"] = amount_predictions
df1["Unique BEL-mentions"] = amount_unique_predictions
df1["Non 1000"] = non_1000_list
df1["Percentage_BEL_mentions"] = round((df1["BEL-mentions"]/df1["n_words"]*100),1)
df1["Percentage_unique_BEL_mentions"] = (
    df1["Unique BEL-mentions"] / df1["BEL-mentions"] * 100
).round(1).fillna(0)

df1["Non 1000 percentage"] = (
    df1["Non 1000"] / df1["Unique BEL-mentions"] * 100
).round(1).fillna(0)
df1= df1.drop("Unnamed: 0", axis=1)

In [ ]:
df1

In [ ]:
df1.to_csv("abel_bel_analysis.csv")
# df1.to_csv("ibis_bel_analysis.csv")
# df1.to_csv("ibis_colon_bel_analysis.csv")

In [ ]:

df_doc = df1[df1["label"]=="Doctor"].reset_index(drop=True)
df_patient = df1[df1["label"]=="Patient"].reset_index(drop=True)

df_doc = df_doc.rename(columns={
    "Percentage_BEL_mentions": "BEL mentions (%)",
    "Percentage_unique_BEL_mentions": "Unique BEL mentions (%)",
    "Non 1000 percentage": "Non-top-1000 words (%)"
})

df_patient = df_patient.rename(columns={
    "Percentage_BEL_mentions": "BEL mentions (%)",
    "Percentage_unique_BEL_mentions": "Unique BEL mentions (%)",
    "Non 1000 percentage": "Non-top-1000 words (%)"
})

metrics = [
    "BEL mentions (%)",
    "Unique BEL mentions (%)",
    "Non-top-1000 words (%)"
]

fig, axes = plt.subplots(1, len(metrics), figsize=(12,4), sharey=False)

for ax, metric in zip(axes, metrics):
    ax.boxplot([df_doc[metric], df_patient[metric]], labels=["Doctor","Patient"])
    ax.set_title(metric)

plt.tight_layout()
plt.show()

## Comparison language

Top 10 words for doctor and patient (corresponding to thesis section "Most used jargon")

In [ ]:
top_arts = (
    df_out[df_out["speaker"] == "Arts"]
    .groupby(["cui_prediction", "prediction"])
    .agg(
        count=("mention", "size"),  
        mentions=("mention", lambda x: ", ".join(sorted(set(x))))  
    )
    .reset_index()
    .sort_values("count", ascending=False)
    .head(10)
)

print(top_arts.to_string())

In [ ]:
top_patient = (
    df_out[df_out["speaker"] == "Patient"]
    .groupby(["cui_prediction", "prediction"])
    .agg(
        count=("mention", "size"),  
        mentions=("mention", lambda x: ", ".join(sorted(set(x))))  
    )
    .reset_index()
    .sort_values("count", ascending=False)
    .head(10)
)

print(top_patient.to_string())

Research category differences (corresponding to thesis section "Most used categories")

In [ ]:
df_doc = df_out[df_out["speaker"]=="Arts"][["cui_prediction"]]
df_patient = df_out[df_out["speaker"]=="Patient"][["cui_prediction"]]
df_doc.rename(columns={'cui_prediction': 'cui'}, inplace=True)
df_patient.rename(columns={'cui_prediction': 'cui'}, inplace=True)

In [ ]:
umls = pd.read_csv("../1_enhance_UMLS/04_ConceptDB/umls-dutch_v1.11.csv")
umls = umls.drop_duplicates(subset="cui")[["cui","type_ids"]]
umls.rename(columns={'type_ids': 'tui'}, inplace=True)
def split_cat(x):
    return x.split("|")
umls["tui"] = umls["tui"].apply(split_cat)
sem_types = pd.read_csv("umls_sem_types.txt")

In [ ]:
df_doc_umls = pd.merge(df_doc,umls,on="cui", how="left")
df_patient_umls = pd.merge(df_patient,umls,on="cui", how="left")

df_doc_counts = df_doc_umls.tui.explode().value_counts(normalize=True)*100
df_patient_counts = df_patient_umls.tui.explode().value_counts(normalize=True)*100
df_doc_counts  = df_doc_counts.round(1)
df_patient_counts = df_patient_counts.round(1)

In [ ]:
tuis_doc = pd.merge(df_doc_counts,sem_types,on="tui", how="left")
tuis_patient = pd.merge(df_patient_counts,sem_types,on="tui", how="left")
doc_10 = tuis_doc.iloc[:10]
patient_10 = tuis_patient.iloc[:10]

Sort different mentions

In [ ]:
df = df_out.copy()
predictions = df_out.groupby("cui_prediction")["prediction"].first().reset_index()
df["mention_norm"] = df["mention"].str.lower().str.strip()

arts_df = df[df["speaker"] == "Arts"]
patient_df = df[df["speaker"] == "Patient"]

arts_counts = arts_df.groupby(["cui_prediction", "mention_norm"]).size().reset_index(name="arts_count")
patient_counts = patient_df.groupby(["cui_prediction", "mention_norm"]).size().reset_index(name="patient_count")

counts = pd.merge(arts_counts, patient_counts, on=["cui_prediction", "mention_norm"], how="outer").fillna(0)
counts[["arts_count", "patient_count"]] = counts[["arts_count", "patient_count"]].astype(int)

counts

In [ ]:
final_records = []

for cui, group in counts.groupby("cui_prediction"):
    doctor_dict = {}
    patient_dict = {}
    shared_dict = {}
    
    for _, row in group.iterrows():
        mention = row["mention_norm"]
        arts_count = row["arts_count"]
        patient_count = row["patient_count"]
        
        if arts_count > 0 and patient_count == 0:
            doctor_dict[mention] = arts_count
        elif patient_count > 0 and arts_count == 0:
            patient_dict[mention] = patient_count
        else:
            shared_dict[mention] = (arts_count, patient_count)
    
    total_count = sum(doctor_dict.values()) + sum(patient_dict.values()) + sum(a + b for a, b in shared_dict.values())
    
    final_records.append({
        "cui_prediction": cui,
        "doctor": doctor_dict,
        "patient": patient_dict,
        "shared": shared_dict,
        "total_count": total_count
    })

final = pd.DataFrame(final_records)
final = final.merge(predictions, on="cui_prediction", how="left")

# Only patient uses
final = final[
    (final["doctor"].apply(len) == 0) &
    (final["patient"].apply(len) > 0) &
    (final["shared"].apply(len) == 0)
]

## Only doctor uses
# final = final[
#     (final["doctor"].apply(len) > 0) &
#     (final["patient"].apply(len) == 0) &
#     (final["shared"].apply(len) == 0)
# ]

In [ ]:
doctor_mentions_list = []
patient_mentions_list = []
shared_mentions_list = []
total_doctor_mentions_list = []
total_patient_mentions_list = []

for _, row in final.iterrows():
    doctor_dict = row["doctor"]
    patient_dict = row["patient"]
    shared_dict = row["shared"]
    

    if doctor_dict:
        sorted_doctor = sorted(doctor_dict.items(), key=lambda x: x[1], reverse=True)
        doctor_mentions_list.append(", ".join(f"{k} ({v})" for k, v in sorted_doctor))
        total_doctor_mentions_list.append(sum(doctor_dict.values()))
    else:
        doctor_mentions_list.append("-")
        total_doctor_mentions_list.append(0)
    

    if patient_dict:
        sorted_patient = sorted(patient_dict.items(), key=lambda x: x[1], reverse=True)
        patient_mentions_list.append(", ".join(f"{k} ({v})" for k, v in sorted_patient))
        total_patient_mentions_list.append(sum(patient_dict.values()))
    else:
        patient_mentions_list.append("-")
        total_patient_mentions_list.append(0)
    
    if shared_dict:
        sorted_shared = sorted(shared_dict.items(), key=lambda x: x[1][0] + x[1][1], reverse=True)
        shared_mentions_list.append(", ".join(f"{k} ({v})" for k, v in sorted_shared))
    else:
        shared_mentions_list.append("-")


final["doctor_mentions"] = doctor_mentions_list
final["patient_mentions"] = patient_mentions_list
final["shared_mentions"] = shared_mentions_list
final["total_doctor_mentions"] = total_doctor_mentions_list
final["total_patient_mentions"] = total_patient_mentions_list

Only doctor uses

In [ ]:
end_table = final[[
    "cui_prediction",
    "doctor_mentions",
    "total_doctor_mentions",
]]

end_table = end_table.sort_values("total_doctor_mentions", ascending=False)
print(end_table.to_string(index=False))

Only patient uses

In [ ]:
end_table = final[[
    "cui_prediction",
    "patient_mentions",
    "total_patient_mentions"
]]

end_table = end_table.sort_values("total_patient_mentions", ascending=False)
print(end_table.to_string(index=False))

Both use

In [ ]:
final = final[
    (final["doctor"].apply(len) > 0) &
    (final["patient"].apply(len) > 0)
]

end_table = final[[
    "cui_prediction",
    "prediction",
    "doctor_mentions",
    "patient_mentions",
    "shared_mentions",
    "total_count",
    "total_doctor_mentions",
    "total_patient_mentions"
]]

end_table = end_table.sort_values("total_doctor_mentions", ascending=False)
print(end_table.to_string(index=False))

In [ ]:
end_table.to_csv("abelcompar.csv")
# end_table.to_csv("ibiscompar.csv")
# end_table.to_csv("ibiscoloncompar.csv")